# Failure annotation / val-split diagnostics

`train_fd_vla.py`와 같은 방식으로 failure cache 파일을 train/val로 나눈 뒤, validation failure 시나리오의

- reason 분포 (`collision` / `offroad` / `goal not reached`)
- collision에서 object head mask 때문에 **구조적으로 맞출 수 없는** 비율

을 계산합니다. split seed, val file 수, reason picker, vehicle mask는 학습 코드와 동일합니다.

In [1]:
from __future__ import annotations

import json
import random
import sys
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_vla" / "train_fd_vla.py").exists():
    REPO_ROOT = Path("/zfsauton2/home/mineuih/waymax_rs")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from data.failure_detection_loader import (
    _cache_scenario_count,
    _has_explicit_scenario_axis,
    _pick_failure_reason,
    _sample_failure_anchor_step,
)
from data.utils import resolve_cache_paths
from train_vla.train_fd_vla import (
    DEFAULT_FAILURE_CACHE_DIR,
    DEFAULT_FAILURE_RESULT_ROOT,
    _split_train_val,
)

print(f"repo root: {REPO_ROOT}")

repo root: /zfsauton2/home/mineuih/waymax_rs


## Config

기본값은 `python train_vla/train_fd_vla.py` argparse와 같습니다.
실제 학습에서 `--failure_cache_dir` / `--failure_result_root`를 바꿨다면 여기도 맞춰 주세요.
per-scenario failure cache는 보통 `exp/<run>/sim_state_cache` 입니다.

In [2]:
FAILURE_CACHE_DIR = "/zfsauton/scratch/mineuih/waymax_rs/exp/simulation_fm_training_collect_failure_20260903_030456/sim_state_cache/"
FAILURE_RESULT_ROOT = "/zfsauton/scratch/mineuih/waymax_rs/exp/simulation_fm_training_collect_failure_20260903_030456/"
FAILURE_FILE_INDICES = None  # e.g. [0, 1, 2] or None for all *.npz

SHUFFLE_SEED = 0
NUM_FAILURE_VAL_FILES = 100
FAILURE_MAX_ANCHOR_OFFSET = 50
VEHICLE_TYPE_ID = 1  # Waymax VEHICLE; used only for type stats, not the object-head mask

print("failure_cache_dir:", FAILURE_CACHE_DIR)
print("failure_result_root:", FAILURE_RESULT_ROOT)
print("shuffle_seed:", SHUFFLE_SEED, "(failure split uses seed+1 =", SHUFFLE_SEED + 1, ")")
print("num_failure_val_files:", NUM_FAILURE_VAL_FILES)

failure_cache_dir: /zfsauton/scratch/mineuih/waymax_rs/exp/simulation_fm_training_collect_failure_20260903_030456/sim_state_cache/
failure_result_root: /zfsauton/scratch/mineuih/waymax_rs/exp/simulation_fm_training_collect_failure_20260903_030456/
shuffle_seed: 0 (failure split uses seed+1 = 1 )
num_failure_val_files: 100


## Train / val file split

`train_fd_vla.py`와 동일합니다.

```python
failure_train_paths, failure_val_paths = _split_train_val(
    failure_cache_dir, failure_file_indices, num_failure_val_files, shuffle_seed + 1
)
```

In [3]:
failure_train_paths, failure_val_paths = _split_train_val(
    FAILURE_CACHE_DIR,
    FAILURE_FILE_INDICES,
    NUM_FAILURE_VAL_FILES,
    SHUFFLE_SEED + 1,
)
all_failure_paths = resolve_cache_paths(FAILURE_CACHE_DIR, FAILURE_FILE_INDICES)

print(f"all failure cache files: {len(all_failure_paths)}")
print(f"train failure files:     {len(failure_train_paths)}")
print(f"val failure files:       {len(failure_val_paths)}")
print("val files:")
for path in failure_val_paths:
    print(" ", Path(path).name)

all failure cache files: 3374
train failure files:     3274
val failure files:       100
val files:
  training_tfexample.tfrecord-00075-of-01000.scenario_031.sim_state_cache.npz
  training_tfexample.tfrecord-00017-of-01000.scenario_197.sim_state_cache.npz
  training_tfexample.tfrecord-00050-of-01000.scenario_424.sim_state_cache.npz
  training_tfexample.tfrecord-00043-of-01000.scenario_208.sim_state_cache.npz
  training_tfexample.tfrecord-00165-of-01000.scenario_243.sim_state_cache.npz
  training_tfexample.tfrecord-00115-of-01000.scenario_137.sim_state_cache.npz
  training_tfexample.tfrecord-00136-of-01000.scenario_260.sim_state_cache.npz
  training_tfexample.tfrecord-00026-of-01000.scenario_376.sim_state_cache.npz
  training_tfexample.tfrecord-00052-of-01000.scenario_241.sim_state_cache.npz
  training_tfexample.tfrecord-00118-of-01000.scenario_278.sim_state_cache.npz
  training_tfexample.tfrecord-00192-of-01000.scenario_436.sim_state_cache.npz
  training_tfexample.tfrecord-00172-of-010

In [4]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def build_failure_result_index(result_root: Path) -> dict[str, Path]:
    if not result_root.exists():
        raise FileNotFoundError(f"failure_result_root does not exist: {result_root}")
    index: dict[str, Path] = {}
    for path in sorted(result_root.rglob("*.json")):
        if path.name == "summary.json":
            continue
        index.setdefault(path.stem, path)
    return index


def result_path_for_cache(cache_file: Path, result_index: dict[str, Path]) -> Path:
    stem = cache_file.name.removesuffix(".sim_state_cache.npz")
    result_path = result_index.get(stem)
    if result_path is not None:
        return result_path
    candidate = cache_file.parent.parent / f"{stem}.json"
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f"Could not find result JSON for {cache_file}")


def select_object_fields(cache_npz: np.lib.npyio.NpzFile, row_idx: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    if _has_explicit_scenario_axis(cache_npz):
        is_sdc = np.asarray(cache_npz["object_metadata_is_sdc"][row_idx]).astype(bool)
        object_types = np.asarray(cache_npz["object_metadata_object_types"][row_idx]).astype(np.int32)
        valid = np.asarray(cache_npz["log_trajectory_valid"][row_idx]).astype(bool)
    else:
        is_sdc = np.asarray(cache_npz["object_metadata_is_sdc"]).astype(bool)
        object_types = np.asarray(cache_npz["object_metadata_object_types"]).astype(np.int32)
        valid = np.asarray(cache_npz["log_trajectory_valid"]).astype(bool)
    if valid.ndim != 2:
        raise ValueError(f"Expected log_trajectory_valid [N, T], got {valid.shape} in {getattr(cache_npz, 'fid', None)}")
    return is_sdc, object_types, valid


def collision_mask_status(
    is_sdc: np.ndarray,
    object_types: np.ndarray,
    valid: np.ndarray,
    gold_object_id: int,
    timestep: int,
) -> dict[str, Any]:
    num_objects, num_steps = valid.shape
    timestep = int(np.clip(timestep, 0, max(0, num_steps - 1)))
    status = {
        "gold_object_id": int(gold_object_id),
        "timestep": timestep,
        "missing_gold": int(gold_object_id) < 0,
        "out_of_range": False,
        "is_ego": False,
        "is_vehicle": False,
        "is_valid": False,
        "in_mask": False,
    }
    if status["missing_gold"]:
        return status
    if gold_object_id >= num_objects:
        status["out_of_range"] = True
        return status
    status["is_ego"] = bool(is_sdc[gold_object_id])
    status["is_vehicle"] = int(object_types[gold_object_id]) == int(VEHICLE_TYPE_ID)
    status["is_valid"] = bool(valid[gold_object_id, timestep])
    status["in_mask"] = (not status["is_ego"]) and status["is_valid"]
    return status


def unanswerable_reason(status: dict[str, Any]) -> str | None:
    if status["in_mask"]:
        return None
    if status["missing_gold"]:
        return "missing_gold_id"
    if status["out_of_range"]:
        return "gold_out_of_range"
    if status["is_ego"]:
        return "gold_is_ego"
    if not status["is_valid"]:
        return "gold_invalid_at_timestep"
    return "unknown"


result_index = build_failure_result_index(Path(FAILURE_RESULT_ROOT))
print(f"indexed result JSONs: {len(result_index)}")

indexed result JSONs: 99995


## Scan validation failure files

각 val cache 파일의 시나리오를 로더와 같이 읽고 `_pick_failure_reason`으로 라벨을 정합니다.
collision이면

- 실패 timestep
- 로더와 같은 방식으로 뽑은 앵커 timestep (`failure_step - offset` ~ `failure_step - 1`)

에서 gold object가 `_other_object_mask`(ego가 아니고 anchor에서 valid)에 들어가는지 봅니다.

In [5]:
def collect_failure_records(cache_paths: tuple[str, ...], rng_seed: int) -> list[dict[str, Any]]:
    paths = list(cache_paths)
    if rng_seed:
        random.Random(int(rng_seed)).shuffle(paths)
    rng = random.Random(int(rng_seed))

    records: list[dict[str, Any]] = []
    for cache_path in paths:
        cache_file = Path(cache_path)
        with np.load(cache_file, allow_pickle=False) as cache_npz:
            num_steps = int(np.asarray(cache_npz["log_trajectory_x"]).shape[-1])
            scenario_count = _cache_scenario_count(cache_npz)
            scenario_order = list(range(scenario_count))
            rng.shuffle(scenario_order)
            for row_idx in scenario_order:
                result_path = result_path_for_cache(cache_file, result_index)
                result = load_json(result_path)
                reason, failure_step = _pick_failure_reason(result, num_steps)
                anchor_step = _sample_failure_anchor_step(
                    failure_step,
                    num_steps,
                    rng,
                    FAILURE_MAX_ANCHOR_OFFSET,
                )
                record = {
                    "cache_path": str(cache_file),
                    "result_path": str(result_path),
                    "row_idx": int(row_idx),
                    "reason": reason,
                    "failure_step": int(failure_step),
                    "anchor_step": int(anchor_step),
                    "gold_object_id": int(result.get("collision_vehicle_id", -1)),
                    "gold_road_id": int(result.get("offroad_road_edge_segment_id", -1)),
                    "json_offroad": bool(result.get("offroad", False)),
                    "json_overlap": bool(result.get("overlap", False)) or bool(result.get("collision", False)),
                    "json_success": bool(result.get("success", False)),
                    "json_goal_reached": bool(result.get("goal_reached", False)),
                }
                if reason == "collision":
                    is_sdc, object_types, valid = select_object_fields(cache_npz, row_idx)
                    window_start = max(0, int(failure_step) - int(FAILURE_MAX_ANCHOR_OFFSET))
                    window_end = max(0, int(failure_step) - 1)
                    window_steps = list(range(window_start, window_end + 1)) if window_end >= window_start else [0]
                    window_in_mask = [
                        collision_mask_status(is_sdc, object_types, valid, record["gold_object_id"], step)["in_mask"]
                        for step in window_steps
                    ]
                    record["failure_mask"] = collision_mask_status(
                        is_sdc, object_types, valid, record["gold_object_id"], failure_step
                    )
                    record["anchor_mask"] = collision_mask_status(
                        is_sdc, object_types, valid, record["gold_object_id"], anchor_step
                    )
                    record["window_steps"] = len(window_steps)
                    record["window_in_mask"] = int(sum(window_in_mask))
                    record["window_unanswerable_frac"] = (
                        1.0 - (record["window_in_mask"] / record["window_steps"]) if record["window_steps"] else 1.0
                    )
                records.append(record)
    return records


val_records = collect_failure_records(failure_val_paths, SHUFFLE_SEED)
print(f"validation failure scenarios: {len(val_records)}")

validation failure scenarios: 100


In [6]:
def pct(numer: int, denom: int) -> str:
    if denom <= 0:
        return "n/a"
    return f"{100.0 * numer / denom:.2f}% ({numer}/{denom})"


reason_counts = Counter(record["reason"] for record in val_records)
print("=== Validation failure reason counts ===")
for reason in ("collision", "offroad", "goal not reached", "success"):
    count = reason_counts.get(reason, 0)
    print(f"{reason:20s} {count:6d}  {pct(count, len(val_records))}")
extra = sorted(set(reason_counts) - {"collision", "offroad", "goal not reached", "success"})
for reason in extra:
    print(f"{reason:20s} {reason_counts[reason]:6d}  {pct(reason_counts[reason], len(val_records))}")
print(f"{'total':20s} {len(val_records):6d}")

collisions = [record for record in val_records if record["reason"] == "collision"]
print("\n=== Collision mask diagnostics ===")
print(f"collision scenarios: {len(collisions)}")

if collisions:
    missing_gold = sum(int(record["anchor_mask"]["missing_gold"]) for record in collisions)
    unanswerable_at_failure = sum(not record["failure_mask"]["in_mask"] for record in collisions)
    unanswerable_at_anchor = sum(not record["anchor_mask"]["in_mask"] for record in collisions)
    annotated = [record for record in collisions if not record["anchor_mask"]["missing_gold"]]
    unanswerable_annotated_anchor = sum(not record["anchor_mask"]["in_mask"] for record in annotated)
    unanswerable_annotated_failure = sum(not record["failure_mask"]["in_mask"] for record in annotated)

    print(f"missing collision_vehicle_id (<0):     {pct(missing_gold, len(collisions))}")
    print(f"unanswerable at failure timestep:      {pct(unanswerable_at_failure, len(collisions))}")
    print(f"unanswerable at sampled anchor:        {pct(unanswerable_at_anchor, len(collisions))}")
    print(f"  among gold_id >= 0, unanswerable @ failure: {pct(unanswerable_annotated_failure, len(annotated))}")
    print(f"  among gold_id >= 0, unanswerable @ anchor:  {pct(unanswerable_annotated_anchor, len(annotated))}")
    print(
        "answerable ceiling @ sampled anchor:   "
        f"{pct(len(collisions) - unanswerable_at_anchor, len(collisions))}"
    )

    failure_reasons = Counter(unanswerable_reason(record["failure_mask"]) or "answerable" for record in collisions)
    anchor_reasons = Counter(unanswerable_reason(record["anchor_mask"]) or "answerable" for record in collisions)
    print("\nwhy unanswerable @ failure timestep:")
    for key, value in failure_reasons.most_common():
        print(f"  {key:28s} {pct(value, len(collisions))}")
    print("why unanswerable @ sampled anchor:")
    for key, value in anchor_reasons.most_common():
        print(f"  {key:28s} {pct(value, len(collisions))}")

    window_fracs = [record["window_unanswerable_frac"] for record in annotated]
    if window_fracs:
        print("\nrandom-anchor window (failure-offset .. failure-1), gold_id >= 0:")
        print(f"  mean unanswerable fraction: {float(np.mean(window_fracs)):.4f}")
        print(f"  median unanswerable fraction: {float(np.median(window_fracs)):.4f}")
        always_unanswerable = sum(frac >= 1.0 - 1e-9 for frac in window_fracs)
        always_answerable = sum(frac <= 1e-9 for frac in window_fracs)
        print(f"  never in mask in the whole window: {pct(always_unanswerable, len(window_fracs))}")
        print(f"  always in mask in the whole window: {pct(always_answerable, len(window_fracs))}")

offroads = [record for record in val_records if record["reason"] == "offroad"]
missing_road = sum(int(record["gold_road_id"] < 0) for record in offroads)
print("\n=== Offroad annotation presence ===")
print(f"offroad scenarios: {len(offroads)}")
print(f"missing offroad_road_edge_segment_id (<0): {pct(missing_road, len(offroads))}")

=== Validation failure reason counts ===
collision                45  45.00% (45/100)
offroad                  46  46.00% (46/100)
goal not reached          9  9.00% (9/100)
success                   0  0.00% (0/100)
total                   100

=== Collision mask diagnostics ===
collision scenarios: 45
missing collision_vehicle_id (<0):     0.00% (0/45)
unanswerable at failure timestep:      0.00% (0/45)
unanswerable at sampled anchor:        2.22% (1/45)
  among gold_id >= 0, unanswerable @ failure: 0.00% (0/45)
  among gold_id >= 0, unanswerable @ anchor:  2.22% (1/45)
answerable ceiling @ sampled anchor:   97.78% (44/45)

why unanswerable @ failure timestep:
  answerable                   100.00% (45/45)
why unanswerable @ sampled anchor:
  answerable                   97.78% (44/45)
  gold_invalid_at_timestep     2.22% (1/45)

random-anchor window (failure-offset .. failure-1), gold_id >= 0:
  mean unanswerable fraction: 0.0222
  median unanswerable fraction: 0.0000
  never in mas

## Optional: exact eval stream

val metric은 success/failure를 한 배치에 섞고 `eval_num_samples`개만 봅니다.
failure-only 전체 val과 그 앞부분 스트림이 얼마나 다른지 보려면 아래를 실행하세요.

In [7]:
EVAL_NUM_SAMPLES = 256
BATCH_SIZE = 8

eval_reason_counts = Counter()
eval_collisions = []
taken = 0
for record in val_records:
    if taken >= EVAL_NUM_SAMPLES // 2:
        break
    eval_reason_counts[record["reason"]] += 1
    if record["reason"] == "collision":
        eval_collisions.append(record)
    taken += 1

print(f"=== First {taken} failure examples in val iterator (eval uses ~{EVAL_NUM_SAMPLES // 2} failures) ===")
for reason in ("collision", "offroad", "goal not reached", "success"):
    count = eval_reason_counts.get(reason, 0)
    print(f"{reason:20s} {count:6d}  {pct(count, taken)}")

if eval_collisions:
    unanswerable = sum(not record["anchor_mask"]["in_mask"] for record in eval_collisions)
    print("\ncollision in this prefix:", len(eval_collisions))
    print("unanswerable @ sampled anchor:", pct(unanswerable, len(eval_collisions)))
    print("answerable ceiling:", pct(len(eval_collisions) - unanswerable, len(eval_collisions)))
    print("If the model is perfect on answerable collisions and 0 on the rest, collision_object_id_accuracy would be that ceiling.")

=== First 100 failure examples in val iterator (eval uses ~128 failures) ===
collision                45  45.00% (45/100)
offroad                  46  46.00% (46/100)
goal not reached          9  9.00% (9/100)
success                   0  0.00% (0/100)

collision in this prefix: 45
unanswerable @ sampled anchor: 2.22% (1/45)
answerable ceiling: 97.78% (44/45)
If the model is perfect on answerable collisions and 0 on the rest, collision_object_id_accuracy would be that ceiling.
